In [1]:
!pip -q install mlxtend
from google.colab import files
up = files.upload()          # choose store_data_association_rules.csv, wait for 100% done
fname = list(up.keys())[0]

import pandas as pd, random
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import fpgrowth, association_rules
from sklearn.model_selection import train_test_split

df = pd.read_csv(fname, header=None)
baskets = [sorted({str(x).strip() for x in r if pd.notna(x) and str(x).strip()}) for r in df.values]
train, test = train_test_split(baskets, test_size=0.2, random_state=42)

te = TransactionEncoder()
oh = pd.DataFrame(te.fit(train).transform(train), columns=te.columns_)
fi = fpgrowth(oh, min_support=0.005, use_colnames=True)
rules = association_rules(fi, metric="confidence", min_threshold=0.20)
rules = rules[rules.lift >= 1.2]
print(len(baskets), len(train), len(test), len(rules))

Saving store_data_association_rules.csv to store_data_association_rules.csv
7501 6000 1501 592


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [2]:
def recommend(items, k=5):
    items = set(items)
    r = rules[rules.antecedents.apply(lambda a: a <= items)]
    out = {}
    for _, row in r.iterrows():
        for c in row.consequents:
            if c not in items and (c not in out or row.lift > out[c][1]):
                out[c] = (row.confidence, row.lift, row.support)
    res = pd.DataFrame([(c, *v) for c, v in out.items()],
                       columns=["product", "confidence", "lift", "support"])
    return res.sort_values(["lift", "confidence"], ascending=False).head(k) if len(res) else res

recommend(["ground beef"])

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

,product,confidence,lift,support
0,spaghetti,0.404075,2.320046,0.039667
2,mineral water,0.429542,1.774965,0.042167
1,milk,0.224109,1.691386,0.022000
3,chocolate,0.235993,1.446332,0.023167


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
random.seed(42)
P = R = H = n = covered = 0
for b in test:
    if len(b) < 2: continue
    b = b[:]; random.shuffle(b)
    h = max(1, len(b)//2)
    inp, rel = b[:h], set(b[h:])
    n += 1
    rec = recommend(inp)
    if len(rec) == 0: continue
    covered += 1
    hit = len(set(rec["product"]) & rel)
    P += hit/len(rec); R += hit/len(rel); H += hit > 0
print(f"Precision@5 {P/covered:.3f} | Recall@5 {R/covered:.3f} | HitRate@5 {H/covered:.3f} | Coverage {covered/n:.3f}")

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

Precision@5 0.147 | Recall@5 0.242 | HitRate@5 0.512 | Coverage 0.914


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packag

In [6]:
import warnings; warnings.filterwarnings("ignore")
import ipywidgets as w
from IPython.display import display, clear_output

pop = pd.Series([i for b in train for i in b]).value_counts()

def recommend_ui(items, k=5):
    r = recommend(items, k)
    if len(r): return r
    top = [p for p in pop.index if p not in items][:k]
    return pd.DataFrame({"product": top, "note": "popular item (no rule found)"})

sel = w.SelectMultiple(options=sorted({i for b in baskets for i in b}), rows=10, description="Basket")
btn = w.Button(description="Recommend"); out = w.Output()
def go(_):
    with out: clear_output(); display(recommend_ui(sel.value))
btn.on_click(go); display(sel, btn, out)

SelectMultiple(description='Basket', options=('almonds', 'antioxydant juice', 'asparagus', 'avocado', 'babies …

Button(description='Recommend', style=ButtonStyle())

Output()